# SocialAuto analytics scratchpad

Ad-hoc queries against `social-postgres`. The `DATABASE_URL` env var is
injected by docker-compose (psycopg2, sync driver).

**First run**: execute the install cell below once, then restart the kernel.

In [ ]:
%pip install -q sqlalchemy psycopg2-binary

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(os.environ["DATABASE_URL"])

def q(sql: str, **params) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

## Recent published posts

In [ ]:
q("""
SELECT substring(p.id::text, 1, 8) AS pid, sa.platform, pt.status,
       pt.platform_url, pt.published_at, left(p.content_text, 60) AS preview
FROM post_targets pt
JOIN posts p ON p.id = pt.post_id
JOIN social_accounts sa ON sa.id = pt.social_account_id
WHERE pt.published_at > now() - interval '7 days'
ORDER BY pt.published_at DESC
""")

## Latest follower counts per account

In [ ]:
q("""
SELECT DISTINCT ON (sa.id) sa.platform, sa.username, sa.account_type,
       fs.followers, fs.captured_at
FROM follower_snapshots fs
JOIN social_accounts sa ON sa.id = fs.social_account_id
ORDER BY sa.id, fs.captured_at DESC
""")

## Engagement by publish hour (Athens) — the timing buckets behind the brief

In [ ]:
q("""
SELECT sa.platform,
       extract(hour FROM p.published_at AT TIME ZONE 'Europe/Athens')::int AS hr,
       count(*) AS posts,
       round(avg(pas.engagement_rate)::numeric, 2) AS avg_er
FROM post_analytics_snapshots pas
JOIN posts p ON p.id = pas.post_id
JOIN post_targets pt ON pt.post_id = p.id AND pt.social_account_id = pas.social_account_id
JOIN social_accounts sa ON sa.id = pas.social_account_id
WHERE pas.captured_at > now() - interval '30 days'
GROUP BY 1, 2 HAVING count(*) >= 2
ORDER BY 1, 4 DESC
""")